# Transformació de les dades de SBD parc agrari: Horari ---> Diari

In [ ]:
import pandas as pd

df = pd.read_csv('/Users/daviddominguez/Documents/VO project Github repository/VO-evolution/Data/Sabadell_Parc_agrari_data.csv')
df.head()

,datetime_utc,temp_mean (°C),temp_max (°C),temp_min (°C),humidity (%),precip (mm),wind_max_10m (km/h),pressure (hPa),radiation (W/m²),station_code,wind_mean_10m (km/h),wind_dir_10m (degrees)
0,2008-10-24 00:00:00,11.9,12.0,11.9,88,0.0,11.2,1022.0,0.0,XF,NaN,NaN
1,2008-10-24 00:30:00,11.9,12.0,11.9,89,0.0,6.1,1022.0,0.0,XF,NaN,NaN
2,2008-10-24 01:00:00,11.8,12.0,11.7,90,0.0,5.0,1022.0,0.0,XF,NaN,NaN
3,2008-10-24 01:30:00,11.6,11.7,11.5,91,0.0,8.6,1022.0,0.0,XF,NaN,NaN
4,2008-10-24 02:00:00,11.7,12.0,11.5,88,0.0,9.0,1021.0,0.0,XF,NaN,NaN


In [3]:
df.drop(columns = ['station_code'], inplace = True)
df.head()

,datetime_utc,temp_mean (°C),temp_max (°C),temp_min (°C),humidity (%),precip (mm),wind_max_10m (km/h),pressure (hPa),radiation (W/m²),wind_mean_10m (km/h),wind_dir_10m (degrees)
0,2008-10-24 00:00:00,11.9,12.0,11.9,88,0.0,11.2,1022.0,0.0,NaN,NaN
1,2008-10-24 00:30:00,11.9,12.0,11.9,89,0.0,6.1,1022.0,0.0,NaN,NaN
2,2008-10-24 01:00:00,11.8,12.0,11.7,90,0.0,5.0,1022.0,0.0,NaN,NaN
3,2008-10-24 01:30:00,11.6,11.7,11.5,91,0.0,8.6,1022.0,0.0,NaN,NaN
4,2008-10-24 02:00:00,11.7,12.0,11.5,88,0.0,9.0,1021.0,0.0,NaN,NaN


In [4]:
import pandas as pd
import numpy as np

# 1. Assegurar datetime
df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], errors="coerce")

# 2. Funció per direcció del vent (important!)
def mean_wind_dir(deg):
    deg = deg.dropna()
    if len(deg) == 0:
        return np.nan
    rad = np.deg2rad(deg)
    sin_mean = np.mean(np.sin(rad))
    cos_mean = np.mean(np.cos(rad))
    return np.rad2deg(np.arctan2(sin_mean, cos_mean)) % 360

# 3. Agregació diària
daily_df = (
    df.groupby(df["datetime_utc"].dt.floor("D"))
      .agg(
          T_avg=("temp_mean (°C)", "mean"),
          T_max=("temp_max (°C)", "max"),
          T_min=("temp_min (°C)", "min"),
          humidity_avg=("humidity (%)", "mean"),
          precip_total=("precip (mm)", "sum"),
          wind_avg=("wind_mean_10m (km/h)", "mean"),
          wind_dir=("wind_dir_10m (degrees)", mean_wind_dir),
          wind_max=("wind_max_10m (km/h)", "max"),
          pressure_avg=("pressure (hPa)", "mean"),
          radiation_avg=("radiation (W/m²)", "mean"),
      )
      .reset_index()
)

# 4. Rename columna data
daily_df = daily_df.rename(columns={"datetime_utc": "date"})


In [5]:
daily_df.tail(20)

,date,T_avg,T_max,T_min,humidity_avg,precip_total,wind_avg,wind_dir,wind_max,pressure_avg,radiation_avg
6349,2026-03-13,11.208333,18.3,4.0,93.020833,0.0,5.129167,241.225375,22.7,1017.195833,216.916667
6350,2026-03-14,8.327083,12.4,3.9,98.770833,21.3,7.975000,280.827931,29.5,1005.237500,29.645833
6351,2026-03-15,9.870833,19.5,1.2,53.125000,0.0,10.033333,323.550382,37.4,1009.114583,239.854167
6352,2026-03-16,10.493750,19.8,0.1,61.895833,0.0,5.637500,321.968451,22.7,1019.275000,232.562500
6353,2026-03-17,10.970833,19.1,3.7,87.750000,0.0,4.350000,222.569996,19.1,1016.439583,233.041667
6354,2026-03-18,10.681250,17.1,2.7,87.437500,0.0,9.912500,93.584309,46.1,1012.493750,238.895833
6355,2026-03-19,9.485417,12.6,2.6,89.500000,0.0,9.139583,45.266094,26.3,1017.737500,65.041667
6356,2026-03-20,8.495833,16.2,1.0,74.666667,0.0,5.595833,196.803926,29.9,1016.583333,243.562500
6357,2026-03-21,8.804167,17.6,0.5,69.208333,0.0,5.068085,247.025442,25.2,1013.256250,244.125000
6358,2026-03-22,10.735417,18.5,2.5,75.354167,0.0,6.914583,260.708637,30.2,1010.750000,217.083333


In [6]:
daily_df['radiation_avg'].isna().sum()

0

In [7]:
daily_df.rename(columns = {'precip_total':'rain_mm', 'wind_avg':'avg_wind_kmh','wind_max':'max_wind_kmh'}, inplace = True)

In [8]:
daily_df.head()

,date,T_avg,T_max,T_min,humidity_avg,rain_mm,avg_wind_kmh,wind_dir,max_wind_kmh,pressure_avg,radiation_avg
0,2008-10-24,15.020833,21.1,9.5,79.062500,0.0,NaN,NaN,20.2,1022.214583,133.458333
1,2008-10-25,17.068750,24.2,11.4,71.520833,0.0,NaN,NaN,26.6,1025.785417,152.375000
2,2008-10-26,15.162500,24.1,9.2,71.291667,0.0,NaN,NaN,20.2,1025.318750,162.833333
3,2008-10-27,16.214583,24.4,9.0,65.479167,0.0,NaN,NaN,30.2,1014.229167,161.916667
4,2008-10-28,12.110417,15.1,9.2,90.166667,27.2,NaN,NaN,17.3,1004.252083,11.812500


In [9]:
nan_percent = daily_df.isna().mean() * 100

print(nan_percent)

date             0.000000
T_avg            0.000000
T_max            0.000000
T_min            0.000000
humidity_avg     0.000000
rain_mm          0.000000
avg_wind_kmh     3.517036
wind_dir         3.517036
max_wind_kmh     0.000000
pressure_avg     0.000000
radiation_avg    0.000000
dtype: float64


In [10]:
daily_df.to_csv('/Users/daviddominguez/Documents/VO project Github repository/VO-evolution/Data/SBD_parc_daily_data.csv', index=False)